In [1]:
%matplotlib inline

import pandas as pd
from PIL import Image
import os
import cv2
from matplotlib import pyplot as plt
from torchvision import  transforms

In [2]:
train_parent_folder = '../../data/Car Labelling/Standford Dataset/train'
test_parent_folder = '../../data/Car Labelling/Standford Dataset/val'

print(os.path.exists(train_parent_folder))
print(os.path.exists(test_parent_folder))

True
True


In [3]:
full_file_path = 'c:/Users/joshu_rdnqgbx/Documents/Current Work/Car Recognition Model/data/Car Labelling/Standford Dataset/train/Acura Integra Type R 2001/00128.jpg'

In [4]:
img = Image.open(full_file_path)

In [5]:
# img.show()
# plt.show()


In [6]:
car_folder_test_arr = 'Acura Integra Type R 2001'.split(' ')

In [7]:
print(car_folder_test_arr[0])
print(' '.join(car_folder_test_arr[1:-1]))
print(car_folder_test_arr[-1])

Acura
Integra Type R
2001


In [8]:
data = []

for car_folder in os.listdir(test_parent_folder):
    car_info = car_folder.split(' ')
    make = car_info[0]
    model = ' '.join(car_info[1:-1])
    year = car_info[-1]
    for car_image in os.listdir(os.path.join(train_parent_folder,car_folder)):
        record = ['train',car_image,make,model,year,car_folder]
        data.append(record)

for car_folder in os.listdir(test_parent_folder):
    car_info = car_folder.split(' ')
    make = car_info[0]
    model = ' '.join(car_info[1:-1])
    year = car_info[-1]
    for car_image in os.listdir(os.path.join(test_parent_folder,car_folder)):
        record = ['val',car_image,make,model,year,car_folder]
        data.append(record)

df = pd.DataFrame(data, columns=['Original_Split','Image', 'Make', 'Model', 'Year','Full_Name'])
df['Make_Encoded'] = df['Make'].astype('category').cat.codes
df['Model_Encoded'] = df['Model'].astype('category').cat.codes
df['Year_Encoded'] = df['Year'].astype('category').cat.codes
df['Full_Name_Encoded'] = df['Full_Name'].astype('category').cat.codes


In [9]:
df.head()

,Original_Split,Image,Make,Model,Year,Full_Name,Make_Encoded,Model_Encoded,Year_Encoded,Full_Name_Encoded
0,train,00128.jpg,Acura,Integra Type R,2001,Acura Integra Type R 2001,1,88,7,1
1,train,00130.jpg,Acura,Integra Type R,2001,Acura Integra Type R 2001,1,88,7,1
2,train,00198.jpg,Acura,Integra Type R,2001,Acura Integra Type R 2001,1,88,7,1
3,train,00255.jpg,Acura,Integra Type R,2001,Acura Integra Type R 2001,1,88,7,1
4,train,00308.jpg,Acura,Integra Type R,2001,Acura Integra Type R 2001,1,88,7,1


In [10]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
import os


class CustomDataset(Dataset):
    def __init__(self, parent_dir, df, transformations, label):
        self.parent_dir = parent_dir
        self.df = df
        self.transformations = transforms.Compose(transformations)
        self.label = label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = self.df.iloc[idx]
        img_split = os.path.join(self.parent_dir, item.Original_Split)
        img_folder = os.path.join(img_split, item.Full_Name)
        img_path = os.path.join(img_folder, item.Image)
        image = Image.open(img_path)
        tensor_image = self.transformations(image)

        label = item[self.label]

        return tensor_image,label

In [39]:
parent_dir = 'c:/Users/joshu_rdnqgbx/Documents/Current Work/Car Recognition Model/data/Car Labelling/Standford Dataset/'

image_size = 384
normalized_mean = [0.485, 0.456, 0.406]
normalized_std = [0.229, 0.224, 0.225]

transformations = []

transformations.append(transforms.Resize((image_size,image_size), interpolation=2))

transformations.append(transforms.ToTensor())
# transformations.append(transforms.Normalize(normalized_mean, normalized_std))


In [41]:
my_dataset = CustomDataset(parent_dir=parent_dir, df = df, transformations=transformations, label = 'Full_Name_Encoded')
train_loader = DataLoader(my_dataset , batch_size=2, shuffle=False, 
                               num_workers=0, drop_last=True)

In [42]:
for batch_features, batch_labels in train_loader:
    # print(batch_features)
    # print(batch_labels
    break

In [20]:
import numpy as np

In [43]:
batch_features[0].permute(1, 2, 0).numpy().shape

(384, 384, 3)

In [44]:
cv2.imshow("Image", batch_features[0].permute(1, 2, 0).numpy())
cv2.waitKey(0)

cv2.destroyAllWindows()